In [ ]:
import numpy as np
import pandas as pd
pd.set_option('display.max_columns', None)
import matplotlib.pyplot as plt
import warnings
import glob
warnings.filterwarnings('ignore')
from jupyterthemes import jtplot
jtplot.style()
from sklearn import metrics

from imblearn.over_sampling import RandomOverSampler
from sklearn.model_selection import train_test_split, KFold, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.ensemble import ExtraTreesClassifier
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.ensemble import AdaBoostClassifier
from sklearn import svm
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix, roc_curve, roc_auc_score, auc
from sklearn.gaussian_process import GaussianProcessClassifier
from sklearn.gaussian_process.kernels import RBF
import random

def pyRandColor():
    randNums = [random.random() for _ in range(0, 3)]

    RGB255 = list([ int(i * 255) for i in randNums ])
    RGB1 = list([ round(i, 2) for i in randNums ])
    return RGB1


## Data

In [ ]:
def getData(filepath, varname='human', cols=['perplexity', 'percentage', 'entropy_rate', 'rates']):
    """
    Load and process a CSV file where the 'rates' column contains stringified lists.

    This follows paper_results003 exactly for rate availability:
    rows with missing rates are dropped; existing rates are read from the CSV
    and expanded into rates_0, rates_1, ... columns.
    """
    import pandas as pd
    import ast

    df = pd.read_csv(filepath)[cols]
    df = df.dropna(subset=["rates"])
    df['rates'] = df['rates'].apply(ast.literal_eval)

    rates_df = df['rates'].apply(pd.Series)
    rates_df.columns = [f'rates_{i}' for i in rates_df.columns]

    df_expanded = pd.concat([df.drop(columns=['rates']), rates_df], axis=1)

    return df_expanded.assign(label=varname)


## Recognition without training using averged estimate of entropy rate

In [ ]:
from pathlib import Path

# Resolve the repository root whether this notebook is run from repo root
# or from the notebooks/ directory.
_CWD = Path.cwd().resolve()
if _CWD.joinpath('api_data_collection').exists():
    REPO_PATH = _CWD
elif _CWD.parent.joinpath('api_data_collection').exists():
    REPO_PATH = _CWD.parent
else:
    raise RuntimeError(
        "Could not locate the nero repository root. Run this notebook from "
        "the repository root or notebooks/ directory."
    )

LEGACY = REPO_PATH / 'paper_data' / 'legacy'
API = REPO_PATH / 'api_data_collection' / 'ai_data'

# Legacy datasets retained from paper_results002.
human1=getData(LEGACY/'gutenberg/gutenberg_table.csv',varname=0)
human2=getData(LEGACY/'arxiv_entropy/arxiv_table.csv',varname=0)
ai1=getData(LEGACY/'ai_longform_gpt4o/ai_web_longform_table.csv',varname=1)
ai2=getData(LEGACY/'ai_longform_gpt3.5/gpt_35_table.csv',varname=1)
ai3=getData(LEGACY/'ai_longform_gpt4.0/gpt_40_table.csv',varname=1)

# API-data tables used by paper_results003.
# As in notebook 003, rates are read directly from each detection CSV.
# Rows with rates == NaN are dropped by getData(); no NERO regeneration occurs.
human3=getData(API/'loc-pd/loc-pd_detection.csv',varname=0)
human4=getData(API/'us-pd/us-pd_detection.csv',varname=0)
ai0=getData(API/'openai/gpt4o/gpt_4o_detection.csv',varname=1)
ai4=getData(API/'claude_sonnet_4/claude_detection.csv',varname=1)
ai5=getData(API/'gemini_2.5_pro/gemini_25_detection.csv',varname=1)
ai6=getData(API/'openai/gpt5/gpt_5_detection.csv',varname=1)
ai7=getData(API/'openai/gpt4.0/gpt_4zero_detection.csv',varname=1)

dallhuman=pd.concat([human1,human2,human3,human4])

df=pd.concat([human1,human2,human3,human4,ai0,ai1,ai2,ai3,ai4,ai5,ai6,ai7])

NAMES={'human1':'Gutenberg project (legacy 002)',
       'human2':'Arxiv papers (legacy 002)',
       'human3':'Library of Congress public domain (003)',
       'human4':'US public domain (003)',
       'ai0':'API GPT-4o (003)',
       'ai1':'AI Generated GPT-4o longform (legacy 002)',
       'ai2':'AI Generated GPT-3.5 longform (legacy 002)',
       'ai3':'AI Generated GPT-4.0 longform (legacy 002)',
       'ai4':'Claude Sonnet 4 (003)',
       'ai5':'Gemini 2.5 Pro (003)',
       'ai6':'GPT-5 (003)',
       'ai7':'API GPT-4.0 (003)',
      }

print('repo:', REPO_PATH)
print('rows after dropping missing rates:', {name: len(globals()[name]) for name in NAMES})


In [ ]:
#dh=df.copy()
df_=df.copy()
df_['nz_entropy_rate']=df[[x for x in df_.columns if 'rates' in x]].replace(0,None).median(axis=1)

def func1(row):
    n=0
    for i in row.values:
        if i == 0:
            n=n+1
    return n
nz=(df!=0).iloc[:,3:43].mean(axis=1)
nzstd=(df!=0).iloc[:,3:43].std(axis=1)
df_['nzmean'] = nz
df_['nzstd'] = nzstd
df_['nzero'] = df.apply(func1,axis=1)
#df=df.drop('nzero',axis=1)

In [ ]:
ytrue=df_.label.values


ypred= df_.fillna(0).nz_entropy_rate.values #+ df.fillna(0).entropy_rate.values
#y_pred=np.median(X,axis=1)
fpr, tpr, thresholds = metrics.roc_curve(ytrue,ypred, pos_label=0)
auc_e=auc(fpr, tpr)
print('nzent',auc_e)
D1=pd.DataFrame({'fpr':fpr,'tpr':tpr,'threshold':thresholds}).set_index('threshold').sort_index().dropna()
#D1.to_csv('./src/neroroce.csv')

ypred=  df_.fillna(0).entropy_rate.values
#y_pred=np.median(X,axis=1)
fpr2, tpr2, thresholds2 = metrics.roc_curve(ytrue,ypred, pos_label=0)
auc_e2=auc(fpr2, tpr2)
print(auc_e2)
D2=pd.DataFrame({'fpr':fpr2,'tpr':tpr2,'threshold':thresholds2}).set_index('threshold').sort_index().dropna()#.to_csv('./src/neroroce.csv')
pd.concat([D1,D2]).to_csv('neroroce.csv')


ypred=  df_.fillna(0).nzero.values
#y_pred=np.median(X,axis=1)
fpr3, tpr3, thresholds3 = metrics.roc_curve(ytrue,ypred, pos_label=0)
auc_e3=auc(fpr3, tpr3)
print(auc_e3)
D3=pd.DataFrame({'fpr':fpr3,'tpr':tpr3,'threshold':thresholds3}).set_index('threshold').sort_index().dropna()#.to_csv('./src/neroroce.csv')
pd.concat([D1,D2,D3]).to_csv('neroroce.csv')



ypred=  df_.fillna(0).perplexity.values
#y_pred=np.median(X,axis=1)
fprP, tprP, thresholdsP = metrics.roc_curve(ytrue,ypred, pos_label=1)
auc_P=auc(fprP, tprP)
print('px',auc_P)
DP=pd.DataFrame({'fpr':fprP,'tpr':tprP,'threshold':thresholdsP}).set_index('threshold').sort_index().dropna()#.to_csv('./src/perplexity.csv')
#D2=pd.DataFrame({'fpr':fpr2,'tpr':tpr2,'threshold':thresholds2}).set_index('threshold').sort_index().dropna()
#pd.concat([D1,D2,DP]).to_csv('./src/neroroce.csv')


ypred=  df_.fillna(0).percentage.values
#y_pred=np.median(X,axis=1)
fprPc, tprPc, thresholdsPc = metrics.roc_curve(ytrue,ypred, pos_label=1)
auc_Pc=auc(fprPc, tprPc)
print('%->',auc_Pc)
DPc=pd.DataFrame({'fpr':fprPc,'tpr':tprPc,'threshold':thresholdsPc}).set_index('threshold').sort_index().dropna()#.to_csv('./src/perplexity.csv')
#D2=pd.DataFrame({'fpr':fpr2,'tpr':tpr2,'threshold':thresholds2}).set_index('threshold').sort_index().dropna()
#pd.concat([D1,D2,DP]).to_csv('./src/neroroce.csv')


In [ ]:
pd.DataFrame(ytrue).to_csv('ytrue.csv')


In [ ]:
from zedstat import zedstat 
zte=zedstat.processRoc(df=pd.read_csv('neroroce.csv'),
           order=3, 
           total_samples=len(ytrue),
           positive_samples=ytrue.sum(),
           alpha=0.01,
           prevalence=.5)

zte.smooth(STEP=0.1,
    interpolate=True,
    convexify=True)
zte.allmeasures(interpolate=True)
zte.usample(precision=2)
zte.getBounds()

auc_e=np.array(zte.auc(alpha=.05))*100
deltae=(auc_e[1]-auc_e[2])/2
le=str(auc_e[0])[:4]+'%'
zte.auc(alpha=.05)

In [ ]:
plt.figure(figsize=(4,4))
ax=plt.gca()
ax=zte.get().tpr.plot(style='--',ax=ax,color='darkorange',lw=2)
#ax=ztz.get().tpr.plot(style='--',ax=ax,color='green',lw=2)
plt.fill_between(x=zte.get().index, y1=zte.df_lim['U'].tpr,y2=zte.df_lim['L'].tpr, alpha=.6)

#plt.fill_between(x=ztz.get().index, y1=ztz.df_lim['U'].tpr,y2=ztz.df_lim['L'].tpr, alpha=.6)
DP.sort_values('tpr').set_index('fpr').tpr.plot(style='--',ax=ax,color='red',lw=2)
DPc.sort_values('tpr').set_index('fpr').tpr.plot(style='--',ax=ax,color='green',lw=2)

#DP.to_csv('perplexity_all.csv')
#DPc.to_csv('percentage_all.csv')

ax.set_xlim(0,1)
ax.set_ylim(0,1)
plt.legend(['auc NERO: '+le],loc='lower right')
ax.set_title('Samples: '+str(len(ytrue))+', AI generated: '+str(int(ytrue.sum())))
ax.set_ylabel('tpr');
plt.savefig('roc-notraining1.pdf',bbox_inches='tight',transparent=True)

In [ ]:
#dx=pd.DataFrame({'tpr_upper':zte.df_lim['U'].tpr,'tpr_lower':zte.df_lim['L'].tpr}).join(zte.get()['tpr'])
#dx.to_csv('nero_all_notraining.csv')

In [ ]:
from sklearn import metrics
df_all=pd.concat([human1,human2,human3,human4,ai0,ai1,ai2,ai3,ai4,ai5,ai6,ai7])
SUBSET='ALL'

In [ ]:
#dh=df.copy()
df_=df_all.copy()
df_['nz_entropy_rate']=df_[[x for x in df_.columns if 'rates' in x]].replace(0,None).median(axis=1)

def func1(row):
    n=0
    for i in row.values:
        if i == 0:
            n=n+1
    return n
nz=(df_!=0).iloc[:,3:43].mean(axis=1)
nzstd=(df_!=0).iloc[:,3:43].std(axis=1)
df_['nzmean'] = nz
df_['nzstd'] = nzstd
df_['nzero'] = df_.apply(func1,axis=1)

In [ ]:
ytrue=df_.label.values
ypred= df_.fillna(0).nz_entropy_rate.values #+ df.fillna(0).entropy_rate.values
#y_pred=np.median(X,axis=1)
fpr, tpr, thresholds = metrics.roc_curve(ytrue,ypred, pos_label=0)
auc_e=auc(fpr, tpr)
print('nzent',auc_e)
D1=pd.DataFrame({'fpr':fpr,'tpr':tpr,'threshold':thresholds}).set_index('threshold').sort_index().dropna()
#D1.to_csv('./src/neroroce.csv')

ypred=  df_.fillna(0).entropy_rate.values
#y_pred=np.median(X,axis=1)
fpr2, tpr2, thresholds2 = metrics.roc_curve(ytrue,ypred, pos_label=0)
auc_e2=auc(fpr2, tpr2)
print(auc_e2)
D2=pd.DataFrame({'fpr':fpr2,'tpr':tpr2,'threshold':thresholds2}).set_index('threshold').sort_index().dropna()#.to_csv('./src/neroroce.csv')
pd.concat([D1,D2]).to_csv('neroroce_frac.csv')


ypred=  df_.fillna(0).nzero.values
#y_pred=np.median(X,axis=1)
fpr3, tpr3, thresholds3 = metrics.roc_curve(ytrue,ypred, pos_label=0)
auc_e3=auc(fpr3, tpr3)
print(auc_e3)
D3=pd.DataFrame({'fpr':fpr3,'tpr':tpr3,'threshold':thresholds3}).set_index('threshold').sort_index().dropna()#.to_csv('./src/neroroce.csv')
pd.concat([D1,D2,D3]).to_csv('neroroce_frac.csv')



ypred=  df_.fillna(0).perplexity.values
#y_pred=np.median(X,axis=1)
fprP, tprP, thresholdsP = metrics.roc_curve(ytrue,ypred, pos_label=1)
auc_P=auc(fprP, tprP)
print('px',auc_P)
DP=pd.DataFrame({'fpr':fprP,'tpr':tprP,'threshold':thresholdsP}).set_index('threshold').sort_index().dropna()#.to_csv('./src/perplexity.csv')
#D2=pd.DataFrame({'fpr':fpr2,'tpr':tpr2,'threshold':thresholds2}).set_index('threshold').sort_index().dropna()
#pd.concat([D1,D2,DP]).to_csv('./src/neroroce.csv')
DP.to_csv('perplexity_frac.csv')


ypred=  df_.fillna(0).percentage.values
#y_pred=np.median(X,axis=1)
fprPc, tprPc, thresholdsPc = metrics.roc_curve(ytrue,ypred, pos_label=1)
auc_Pc=auc(fprPc, tprPc)
print('%->',auc_Pc)
DPc=pd.DataFrame({'fpr':fprPc,'tpr':tprPc,'threshold':thresholdsPc}).set_index('threshold').sort_index().dropna()#.to_csv('./src/perplexity.csv')
DPc = DPc[~np.isinf(DPc.index)]
DPc.to_csv('percentage_frac.csv')
#D2=pd.DataFrame({'fpr':fpr2,'tpr':tpr2,'threshold':thresholds2}).set_index('threshold').sort_index().dropna()
#pd.concat([D1,D2,DP]).to_csv('./src/neroroce.csv')


In [ ]:
from zedstat import zedstat 
zte=zedstat.processRoc(df=pd.read_csv('neroroce_frac.csv'),
           order=3, 
           total_samples=len(ytrue),
           positive_samples=ytrue.sum(),
           alpha=0.01,
           prevalence=.5)

zte.smooth(STEP=0.001)
zte.allmeasures(interpolate=True)
zte.usample(precision=2)
zte.getBounds()

auc_e=np.array(zte.auc(alpha=.05))*100
deltae=(auc_e[1]-auc_e[2])/2
le=str(auc_e[0])[:4]+'%'
zte.auc(alpha=.05)


ztpc=zedstat.processRoc(df=pd.read_csv('percentage_frac.csv'),
           order=3, 
           total_samples=len(ytrue),
           positive_samples=ytrue.sum(),
           alpha=0.01,
           prevalence=.5)

ztpc.smooth(STEP=0.001)
ztpc.allmeasures(interpolate=True)
ztpc.usample(precision=2)
ztpc.getBounds()

auc_pc=np.array(ztpc.auc(alpha=.05))*100
deltapc=(auc_pc[1]-auc_pc[2])/2
lpc=str(auc_pc[0])[:4]+'%'
ztpc.auc(alpha=.05)



ztpx=zedstat.processRoc(df=pd.read_csv('perplexity_frac.csv'),
           order=3, 
           total_samples=len(ytrue),
           positive_samples=ytrue.sum(),
           alpha=0.01,
           prevalence=.5)

ztpx.smooth(STEP=0.001)
ztpx.allmeasures(interpolate=True)
ztpx.usample(precision=2)
ztpx.getBounds()

auc_px=np.array(ztpx.auc(alpha=.05))*100
deltapx=(auc_px[1]-auc_px[2])/2
lpx=str(auc_px[0])[:4]+'%'
ztpx.auc(alpha=.05)

In [ ]:
plt.figure(figsize=(4,4))
ax=plt.gca()
ax=zte.get().tpr.plot(style='--',ax=ax,color='darkorange',lw=2)
ax=ztpc.get().tpr.plot(style='--',ax=ax,color='green',lw=2)
ax=ztpx.get().tpr.plot(style='--',ax=ax,color='red',lw=2)

plt.fill_between(x=zte.get().index, y1=zte.df_lim['U'].tpr,y2=zte.df_lim['L'].tpr, alpha=.6)
plt.fill_between(x=ztpc.get().index, y1=ztpc.df_lim['U'].tpr,y2=ztpc.df_lim['L'].tpr, alpha=.6)
plt.fill_between(x=ztpx.get().index, y1=ztpx.df_lim['U'].tpr,y2=ztpx.df_lim['L'].tpr, alpha=.6)


dx=pd.DataFrame({'tpr_upper':zte.df_lim['U'].tpr,'tpr_lower':zte.df_lim['L'].tpr}).join(zte.get()['tpr'])
dx.to_csv('nero_'+SUBSET+'_notraining.csv')
dx=pd.DataFrame({'tpr_upper':ztpc.df_lim['U'].tpr,'tpr_lower':ztpc.df_lim['L'].tpr}).join(ztpc.get()['tpr'])
dx.to_csv('percentage_'+SUBSET+'.csv')
dx=pd.DataFrame({'tpr_upper':ztpx.df_lim['U'].tpr,'tpr_lower':ztpx.df_lim['L'].tpr}).join(ztpx.get()['tpr'])
dx.to_csv('perplexity_'+SUBSET+'.csv')


ax.set_xlim(0,1)
ax.set_ylim(0,1)
plt.legend(['auc NERO: '+le,'auc %: '+lpc,'auc perplx: '+lpx],loc='lower right')
ax.set_title('Samples: '+str(len(ytrue))+', AI generated: '+str(int(ytrue.sum())))
ax.set_ylabel('tpr');
plt.savefig('roc-notraining'+SUBSET+'.pdf',bbox_inches='tight',transparent=True)

# NERO classifier: human vs GPT-5

**paper_results007 change:** the subset/triplet robustness block has been removed to make the notebook substantially faster.

This classifier experiment uses only:

- **Human class (0):** all four human corpora `human1`–`human4`
- **AI class (1):** GPT-5 only (`ai6`)

The combined human+GPT-5 cohort is split 50/50 into train and test sets with `stratify=y` and `random_state=42`.

The classifier uses all **46 NERO-derived features**:

- `entropy_rate`
- `rates_0` through `rates_40`
- `nz_entropy_rate`
- `nzmean`
- `nzstd`
- `nzero`

The four derived features are calculated without using the label column, so there is no target leakage.


In [ ]:
from sklearn.model_selection import train_test_split

def add_classifier_nero_features(frame):
    """
    Add the four derived NERO features used by the classifier.
    These are computed only from NERO inputs; label is never used.
    """
    out = frame.copy()

    rate_cols = sorted(
        [c for c in out.columns if c.startswith('rates_')],
        key=lambda c: int(c.split('_')[1])
    )
    base_nero_cols = ['entropy_rate'] + rate_cols

    # Median of nonzero rate-vector components.
    out['nz_entropy_rate'] = out[rate_cols].replace(0, np.nan).median(axis=1)

    # Fraction and standard deviation of nonzero rate-vector components.
    nz_mask = out[rate_cols].ne(0) & out[rate_cols].notna()
    out['nzmean'] = nz_mask.mean(axis=1)
    out['nzstd'] = nz_mask.astype(float).std(axis=1)

    # Count zeros only among NERO inputs (never across label).
    out['nzero'] = out[base_nero_cols].eq(0).sum(axis=1)

    return out


# paper_results007: ONLY human data + GPT-5 enter the classifier experiment.
df_classifier = pd.concat(
    [human1, human2, human3, human4, ai6],
    axis=0,
    ignore_index=True
)
df_classifier = add_classifier_nero_features(df_classifier)

rate_cols = sorted(
    [c for c in df_classifier.columns if c.startswith('rates_')],
    key=lambda c: int(c.split('_')[1])
)

feature_cols = (
    ['entropy_rate']
    + rate_cols
    + ['nz_entropy_rate', 'nzmean', 'nzstd', 'nzero']
)

assert len(rate_cols) == 41, f"Expected 41 rates_* columns, found {len(rate_cols)}"
assert len(feature_cols) == 46, f"Expected 46 classifier features, found {len(feature_cols)}"

X = df_classifier[feature_cols].values
y = df_classifier['label'].astype(int).values

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.5,
    random_state=42,
    stratify=y
)

print("Classifier cohort")
print("  Human samples:", int((y == 0).sum()))
print("  GPT-5 samples:", int((y == 1).sum()))
print("  Total samples:", len(y))
print("  Train samples:", len(y_train),
      f"(human={(y_train == 0).sum()}, GPT-5={(y_train == 1).sum()})")
print("  Test samples :", len(y_test),
      f"(human={(y_test == 0).sum()}, GPT-5={(y_test == 1).sum()})")
print("  Features     :", len(feature_cols))


In [ ]:
%%time
from sklearn import metrics
from sklearn.impute import SimpleImputer

# Fit preprocessing on the training set only.
imputer = SimpleImputer(strategy='constant', fill_value=0)
X_train = imputer.fit_transform(X_train)
X_test = imputer.transform(X_test)

modelnames = [
    'randomforest',
    'gaussianp'
]

models = [
    RandomForestClassifier(
        n_estimators=2500,
        class_weight='balanced',
        random_state=42
    ).fit(X_train, y_train),

    GaussianProcessClassifier(
        kernel=1.0 * RBF(1.0),
        random_state=42
    ).fit(X_train, y_train)
]

y_preds = [model.predict_proba(X_test) for model in models]

auc_scores = {}
ROC = {}

for modelname, y_pred in zip(modelnames, y_preds):
    fpr, tpr, thresholds = metrics.roc_curve(
        y_test,
        y_pred[:, 1],
        pos_label=1
    )
    auc_scores[modelname] = auc(fpr, tpr)
    ROC[modelname] = {
        'fpr': fpr,
        'tpr': tpr,
        'thresholds': thresholds
    }

print("Human vs GPT-5 test AUCs:")
print(auc_scores)


In [ ]:
plt.figure(figsize=[20,10])

assert len(models[0].feature_importances_) == 46
assert len(feature_cols) == 46

pd.Series(
    models[0].feature_importances_,
    index=feature_cols,
    name='feature_importance'
).plot(ax=plt.gca(), kind='bar')

plt.title('Random Forest feature importance: human vs GPT-5')
plt.ylabel('Feature importance')
plt.tight_layout()


In [ ]:
modeldf_ = [pd.DataFrame(ROC[model_]) for model_ in modelnames]

ztc = zedstat.processRoc(
    df=modeldf_[-1],
    order=3,
    total_samples=len(y_test),
    positive_samples=y_test.sum(),
    alpha=0.01,
    prevalence=.5
)

ztc.smooth(STEP=0.001)
ztc.allmeasures(interpolate=True)
ztc.usample(precision=2)
ztc.getBounds()

auc_c = np.array(ztc.auc(alpha=.05)) * 100
deltac = (auc_c[1] - auc_c[2]) / 2
lc = str(auc_c[0])[:4] + '%'

print("Gaussian-process NERO classifier AUC with bounds:")
ztc.auc(alpha=.05)


In [ ]:
plt.figure(figsize=(4,4))
ax = plt.gca()

# Classifier ROC for the human-vs-GPT5 experiment.
ax = ztc.get().tpr.plot(style='-', ax=ax, lw=2)
plt.fill_between(
    x=ztc.get().index,
    y1=ztc.df_lim['U'].tpr,
    y2=ztc.df_lim['L'].tpr,
    alpha=.35
)

ax.set_xlim(0,1)
ax.set_ylim(0,1)
plt.legend(['NERO classifier AUC: ' + lc], loc='lower right')
ax.set_title(
    'Human vs GPT-5\n'
    + 'Test samples: ' + str(len(y_test))
    + ', GPT-5: ' + str(int(y_test.sum()))
)
ax.set_xlabel('FPR')
ax.set_ylabel('TPR')
plt.tight_layout()
plt.savefig(
    'roc-human-vs-gpt5-classifier.pdf',
    bbox_inches='tight',
    transparent=True
)
